# Qwen3-30B-A3B MI355X — old vs fixed collection: what changed, and why

Compares two `linear_op.csv` generations for the same model/grid, to separate two different
causes of any difference: **a different collection run** (new node, new date, ordinary
run-to-run variance) vs **a methodology fix** (two-pass GPU-bound timing, fused RoPE kernel,
GC disabled). This notebook does not modify, and is not read by, either of the other two:
`linear_ops_run_groups_and_stats.ipynb` (old data alone) or
`linear_ops_dense_fixed_run_groups_and_stats.ipynb` (new data alone, plus its own
GPU-bound-vs-legacy comparison *within* the new run only).

## The two datasets

| | Old | New |
|---|---|---|
| Collected | 2026-09-15/16 | 2026-09-22 |
| Job | 21313 / 21334 | 21483 |
| Node | `amd-mi355x-9` / `amd-mi355x-8` | `amd-mi355x-1` |
| Timed runs/shape | 50 (`emb`: 100) | 25 (`emb`: 50) — halved by the queue-backpressure knob |
| `attn_rope` implementation | pure-PyTorch fallback (wrong API path; also numerically not vLLM's rotation, see `07_` S3.5) | `vllm_kernel` (fused, correct) |
| Timing method | single CUDA-event pair per scope; host-bound whenever the GPU catches up | two-pass: `time_stats_hostbound.*` (same as old's method) and `time_stats.*` (GPU held behind the host by a spin — GPU-bound) |
| `attn_pre_proj` run-11 GC stall | present (masked in the old notebook) | fixed at the source (`gc.disable()` around the timed loop) |
| Canonical / staged? | yes, `data/profiling/compute/mi355x/qwen3-a3b-30b-moe/linear_op.csv` | not yet — this notebook reads the same checksummed local copy the `dense_fixed` notebook uses |

**Why a three-way split, not a single before/after ratio.** The new run changed *two* things
at once: it's a different collection (different node/date/ordinary noise) *and* it uses a fixed
methodology. The new file's own `time_stats_hostbound.<op>` column is measured the *same way*
the old file was (single-pass, host-bound-prone) but on the *new* run — so:

- **old vs new-legacy** (`time_stats_hostbound`) isolates what changed **just from re-collecting**
  (same method, different run) — should be close to 1.0 outside ordinary noise.
- **new-legacy vs new-GPU-bound** (`time_stats_hostbound` vs `time_stats`, both from the new run)
  isolates what changed **just from the fix** (same run, different method) — this is exactly the
  `dense_fixed` notebook's own comparison, repeated here for the decomposition.
- **old vs new-GPU-bound** is the number a user actually sees swapping datasets — the product of
  both effects, not attributable to either alone without the two ratios above.

**Caveats specific to this comparison:**
- `attn_rope`'s ratios conflate the timing-methodology fix with a genuine kernel/numerics change
  (torch fallback -> vLLM kernel) — a large old-vs-new difference there is not purely a
  measurement artifact.
- The new run has half the timed samples per shape (25 vs 50, or 50 vs 100 for `emb`), so its
  medians carry more run-to-run noise than the old file's, independent of any real effect.
- `forward_gpu_span` exists only in the new file (no old counterpart) — not compared here.
- Run-group (within-shape run-position) comparisons are not attempted across datasets: bin sizes
  differ (10 vs 5 runs/group) and the timing method itself changed, so a run-group-level diff
  would not isolate anything meaningful.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OLD_PATH = "/home/dn/Frontier-qwen3-profiling/data/profiling/compute/mi355x/qwen3-a3b-30b-moe/linear_op.csv"
NEW_PATH = "/tmp/claude-1001/-home-dn-amd-playground/dedce0c4-4547-4649-84a2-ad89dca651aa/scratchpad/dense_fixed/linear_op.csv"

old = pd.read_csv(OLD_PATH, low_memory=False, float_precision="round_trip")
new = pd.read_csv(NEW_PATH, low_memory=False, float_precision="round_trip")

OPS = ["attn_pre_proj", "attn_post_proj", "attn_rope",
       "input_layernorm", "post_attention_layernorm", "emb"]

CATEGORICAL_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
                      "#008300", "#4a3aa7", "#e34948"]
OLD_COLOR = "#898781"     # muted grey — the reference being compared against
NEW_GPU_COLOR = "#2a78d6"  # blue — the fixed, GPU-bound measurement
NEW_LEGACY_COLOR = "#eb6834"  # orange — same run as NEW_GPU, old-style method

SURFACE = "#fcfcfb"
GRID = "#e1e0d9"
AXIS_LINE = "#c3c2b7"
TEXT_PRIMARY = "#0b0b0b"

print("old:", old.shape, "| new:", new.shape)
print("old num_tokens:", old["num_tokens"].nunique(), "| new num_tokens:", new["num_tokens"].nunique())
print("old timed runs (attn_pre_proj):", int(old["time_stats.attn_pre_proj.count"].dropna().iloc[0]),
      "| new:", int(new["time_stats.attn_pre_proj.count"].dropna().iloc[0]))


old: (13308, 69) | new: (13308, 148)
old num_tokens: 3327 | new num_tokens: 3327
old timed runs (attn_pre_proj): 50 | new: 25


In [2]:
def tp_values_with_data(df, op):
    col = f"time_stats.{op}.mean"
    present = df.loc[df[col].notna(), "num_tensor_parallel_workers"].unique()
    return sorted(int(t) for t in present)


def comparison_frame(op, tp):
    """(num_tokens, old_median, new_legacy_median, new_gpu_median) inner-joined on
    num_tokens for this op/TP, sorted by num_tokens."""
    o = old[(old.num_tensor_parallel_workers == tp) & old[f"time_stats.{op}.median"].notna()]
    n = new[(new.num_tensor_parallel_workers == tp) & new[f"time_stats.{op}.median"].notna()]
    o = o[["num_tokens", f"time_stats.{op}.median"]].rename(columns={f"time_stats.{op}.median": "old_median"})
    n = n[["num_tokens", f"time_stats.{op}.median", f"time_stats_hostbound.{op}.median"]].rename(
        columns={f"time_stats.{op}.median": "new_gpu_median",
                 f"time_stats_hostbound.{op}.median": "new_legacy_median"})
    d = o.merge(n, on="num_tokens", how="inner").sort_values("num_tokens")
    return d


def plot_overlay(op, tp):
    d = comparison_frame(op, tp)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d.num_tokens, y=d.old_median, mode="lines", name="Old (2026-09-15/16)",
                              line=dict(color=OLD_COLOR, width=2, dash="dot")))
    fig.add_trace(go.Scatter(x=d.num_tokens, y=d.new_legacy_median, mode="lines",
                              name="New, legacy method (time_stats_hostbound)",
                              line=dict(color=NEW_LEGACY_COLOR, width=2, dash="dash")))
    fig.add_trace(go.Scatter(x=d.num_tokens, y=d.new_gpu_median, mode="lines",
                              name="New, GPU-bound (time_stats)",
                              line=dict(color=NEW_GPU_COLOR, width=2, dash="solid")))
    fig.update_xaxes(title_text="num_tokens (shape)", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_yaxes(title_text="median time (ms), log scale", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_layout(
        title=f"{op} (TP {tp}) — old vs new-legacy vs new-GPU-bound",
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, font=dict(color=TEXT_PRIMARY),
        height=500, width=900,
    )
    return fig


def plot_ratio_decomposition(op, tp):
    d = comparison_frame(op, tp)
    recollection_only = d.new_legacy_median / d.old_median
    fix_only = d.new_gpu_median / d.new_legacy_median
    net = d.new_gpu_median / d.old_median
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d.num_tokens, y=recollection_only, mode="lines",
                              name="recollection only (new-legacy / old)",
                              line=dict(color=OLD_COLOR, width=2)))
    fig.add_trace(go.Scatter(x=d.num_tokens, y=fix_only, mode="lines",
                              name="fix only (new-GPU-bound / new-legacy)",
                              line=dict(color=NEW_LEGACY_COLOR, width=2)))
    fig.add_trace(go.Scatter(x=d.num_tokens, y=net, mode="lines",
                              name="net (new-GPU-bound / old)",
                              line=dict(color=NEW_GPU_COLOR, width=2, dash="dot")))
    fig.add_hline(y=1.0, line=dict(color=AXIS_LINE, width=1, dash="dot"))
    fig.update_xaxes(title_text="num_tokens (shape)", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_yaxes(title_text="ratio, log scale", type="log", gridcolor=GRID, linecolor=AXIS_LINE)
    fig.update_layout(
        title=f"{op} (TP {tp}) — ratio decomposition: recollection vs fix vs net",
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, font=dict(color=TEXT_PRIMARY),
        height=500, width=900,
    )
    return fig


## `attn_pre_proj`

In [3]:
tps = sorted(set(tp_values_with_data(old, "attn_pre_proj")) & set(tp_values_with_data(new, "attn_pre_proj")))
for tp in tps:
    plot_overlay("attn_pre_proj", tp).show()
    plot_ratio_decomposition("attn_pre_proj", tp).show()

## `attn_post_proj`

In [4]:
tps = sorted(set(tp_values_with_data(old, "attn_post_proj")) & set(tp_values_with_data(new, "attn_post_proj")))
for tp in tps:
    plot_overlay("attn_post_proj", tp).show()
    plot_ratio_decomposition("attn_post_proj", tp).show()

## `attn_rope`

**Confound.** `attn_rope` changed implementation between these two datasets (torch fallback -> `vllm_kernel`), not just measurement method — its ratios below mix a real kernel/numerics change with the timing-methodology fix. Do not read `fix only` here as measurement bias alone.

In [5]:
tps = sorted(set(tp_values_with_data(old, "attn_rope")) & set(tp_values_with_data(new, "attn_rope")))
for tp in tps:
    plot_overlay("attn_rope", tp).show()
    plot_ratio_decomposition("attn_rope", tp).show()

## `input_layernorm`

In [6]:
tps = sorted(set(tp_values_with_data(old, "input_layernorm")) & set(tp_values_with_data(new, "input_layernorm")))
for tp in tps:
    plot_overlay("input_layernorm", tp).show()
    plot_ratio_decomposition("input_layernorm", tp).show()

## `post_attention_layernorm`

In [7]:
tps = sorted(set(tp_values_with_data(old, "post_attention_layernorm")) & set(tp_values_with_data(new, "post_attention_layernorm")))
for tp in tps:
    plot_overlay("post_attention_layernorm", tp).show()
    plot_ratio_decomposition("post_attention_layernorm", tp).show()

## `emb`

In [8]:
tps = sorted(set(tp_values_with_data(old, "emb")) & set(tp_values_with_data(new, "emb")))
for tp in tps:
    plot_overlay("emb", tp).show()
    plot_ratio_decomposition("emb", tp).show()